# 07 — Benchmarking with lm-evaluation-harness

**lm-eval** runs a model against standardized tasks (GSM8K, IFEval, GPQA, …) and reports
metrics. One function — `lm_eval.simple_evaluate` — drives every backend; you choose the
backend and pass its `model_args`.

**Library focus:** `lm_eval.simple_evaluate`, `lm_eval.tasks.TaskManager`.

> `limit=N` runs only N items/task — use it to smoke-test before a full battery.

In [ ]:
MODEL = "Qwen/Qwen3.5-4B"

## vLLM backend (fast at scale)

`model="vllm"` with single-GPU `model_args`. `enable_thinking=False` keeps a reasoning
model's answer within the extraction budget; `apply_chat_template=True` for instruct models.

In [ ]:
import lm_eval

results = lm_eval.simple_evaluate(
    model="vllm",
    model_args={
        "pretrained": MODEL,
        "tensor_parallel_size": 1,
        "dtype": "auto",
        "gpu_memory_utilization": 0.9,
        "enable_thinking": False,
    },
    tasks=["gsm8k"],
    limit=10,
    batch_size="auto",
    apply_chat_template=True,
)
print(results["results"]["gsm8k"])

## HF backend

`model="hf"`. On WSL use a **fixed** `batch_size` (not `"auto"`, which can thrash here).

In [ ]:
results_hf = lm_eval.simple_evaluate(
    model="hf",
    model_args={"pretrained": MODEL, "dtype": "bfloat16"},
    tasks=["gsm8k"],
    limit=10,
    batch_size=8,  # fixed batch on WSL
    apply_chat_template=True,
)
print(results_hf["results"]["gsm8k"])

## Reading the results

`results["results"]` maps task -> {`metric,filter`: value}. A small flattener gives clean
`task/metric -> value` rows (the shape `scripts/forgetting_report.py` diffs against base).

In [ ]:
def flatten(results):
    rows = {}
    for task, metrics in results["results"].items():
        for key, val in metrics.items():
            if (
                key == "alias"
                or ",none" not in key
                and not isinstance(val, (int, float))
            ):
                continue
            if "stderr" in key:
                continue
            metric = key.split(",")[0]
            if isinstance(val, (int, float)):
                rows[f"{task}/{metric}"] = float(val)
    return rows


print(flatten(results))

## Custom tasks & base + LoRA (reference)

Custom task YAMLs not shipped upstream (SuperGPQA, IFBench) load via a `TaskManager`. To
evaluate a fine-tuned model without merging, pass the adapter path in `model_args` — vLLM
uses `lora_local_path`, the HF backend uses `peft`. (Commented — needs the
notebook-05 adapter.)

```python
from lm_eval.tasks import TaskManager
tm = TaskManager(include_path="eval_tasks/supergpqa")
lm_eval.simple_evaluate(model="vllm", model_args={...}, tasks=["supergpqa"],
                        task_manager=tm, limit=5)

# base + LoRA adapter (vLLM):
lm_eval.simple_evaluate(model="vllm", tasks=["gsm8k"], limit=10, apply_chat_template=True,
    model_args={"pretrained": MODEL, "tensor_parallel_size": 1, "dtype": "auto",
                "lora_local_path": "runs/_examples_05_lora/adapter", "max_lora_rank": 16})
```

> **Project pointer:** `scripts/evaluate.py` wraps this (config-driven batteries in
> `configs/eval/`, the profile-derived thinking/chat policy, custom-task include paths);
> `llm_core.evaluation.summarize` / `flatten_results` produce the rows shown above.